In [1]:
import sys

import polars as pl
import torch

from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


In [2]:
target_dyn_demand_weekly = pl.read_parquet(DIR + 'target_dyn_demand_weekly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_weekly

oper_part_no,demand_dt,demand_qty
str,i64,f64
"""0001-1001""",201811,5.0
"""0001-1001""",201847,7.0
"""0001-1001""",202005,20.0
"""0001-1001""",202006,100.0
"""0001-1001""",202010,2.0
…,…,…
"""ZZ90239""",202043,1.0
"""ZZ90239""",202145,1.0
"""ZZ90239""",202325,1.0


In [3]:
plan_yyyymm = 201811
lookback = 54
horizon = 27

data_module = MultiPartDataModule(
    target_dyn_demand_weekly.sort(['oper_part_no', 'demand_dt']),
    lookback = lookback,
    horizon = horizon,
    batch_size = 64,
    val_ratio = 0.2,
    is_running = True
)

train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [ ]:
from modeling_module.training.model_trainers.total_train import run_total_train_weekly

mode_dict = run_total_train_weekly(
    train_loader,
    val_loader,
    lookback = lookback,
    horizon = horizon,
)

[EXO] base exo_dim=2 exo_head? True
[EXO] qmdl exo_dim=2 exo_head? True
PatchMixer Base (Weekly)
[EXO-batch] exo normalized to shape=(64, 27, 2) (expect [B,H,E])


C:\Users\USER\python\py312\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1028.)
  return F.conv1d(


Epoch 1/50 | LR 0.000976 | Train 113123.415162 | Val 107013.246737
Epoch 2/50 | LR 0.000905 | Train 113034.224822 | Val 107227.030401
Epoch 3/50 | LR 0.000794 | Train 113013.358424 | Val 106981.760265
Epoch 4/50 | LR 0.000655 | Train 112974.172474 | Val 107220.475021
Epoch 5/50 | LR 0.000500 | Train 113040.852301 | Val 107287.282519
Epoch 6/50 | LR 0.000345 | Train 112858.556974 | Val 107076.409282
Epoch 7/50 | LR 0.000206 | Train 112746.539698 | Val 107082.834438
Epoch 8/50 | LR 0.000095 | Train 112646.547317 | Val 107051.305733
Epoch 9/50 | LR 0.000024 | Train 112480.595718 | Val 107005.693330
Epoch 10/50 | LR 0.000000 | Train 112546.875233 | Val 107096.989685
Epoch 11/50 | LR 0.000024 | Train 112360.375212 | Val 107091.523971
Epoch 12/50 | LR 0.000095 | Train 112463.254509 | Val 107087.146568
Epoch 13/50 | LR 0.000206 | Train 112418.860228 | Val 107035.853629
Epoch 14/50 | LR 0.000345 | Train 112345.292375 | Val 107004.046600
Epoch 15/50 | LR 0.000500 | Train 112436.608175 | Val 106

In [ ]:
from modeling_module.utils.checkpoint import save_model_dict, load_model_dict
from modeling_module.utils.exogenous_utils import calendar_sin_cos
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfigWeekly

save_dir = DIR + 'fit/20251104_running'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

pm_base_config = PatchMixerConfigWeekly(
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

pm_quantile_config = PatchMixerConfigWeekly(
    device = device,
    loss_mode = 'quantile',
    quantiles = (0.1, 0.5, 0.9)
)

# ti_config = TitanConfigMonthly(
#         device = device,
#         loss_mode = 'point',
#         point_loss = 'mae'
#     )
#
# ti_patch_config = TitanConfigPatchMonthly(
#     device = device,
#     loss_mode = 'point',
#     point_loss = 'mae'
# )
#
# pt_config = PatchTSTConfigMonthly(
#         device = device,
#         loss_mode = 'auto',
#         quantiles = (0.1, 0.5, 0.9)
#     )

cfg_map = {
    "PatchMixer Base": pm_base_config,
    "PatchMixer Quantile": pm_quantile_config,
    # "Titan Base": ti_config,
    # "Titan LMM": ti_config,
    # "Titan Seq2Seq": ti_config,
    # "Titan Patch": ti_patch_config,
    # "PatchTST Base": pt_config,
    # "PatchTST Quantile": pt_config
}

builder_key_by_name = {
  "PatchMixer Base": "patchmixer_base",
  "PatchMixer Quantile": "patchmixer_quantile",
  # "Titan Base": "titan_base",
  # "Titan LMM": "titan_lmm",
  # "Titan Seq2Seq": "titan_seq2seq",
  # "Titan Patch": "titan_patch",
  # "PatchTST Base": "patchtst_base",
  # "PatchTST Quantile": "patchtst_quantile",
}
save_index = save_model_dict(mode_dict, save_dir, cfg_by_name = cfg_map, builder_key_by_name=builder_key_by_name)

# Load
from modeling_module.models.model_builder import (
    build_patch_mixer_base, build_patch_mixer_quantile,
)

builders = {
    "patchmixer_base": lambda cfg: build_patch_mixer_base(cfg or PatchMixerConfigWeekly()),
    "patchmixer_quantile": lambda cfg: build_patch_mixer_quantile(cfg or PatchMixerConfigWeekly()),
    # "titan_base": lambda cfg: build_titan_base(cfg or TitanConfigMonthly()),
    # "titan_lmm": lambda cfg: build_titan_lmm(cfg or TitanConfigMonthly()),
    # "titan_seq2seq": lambda cfg: build_titan_seq2seq(cfg or TitanConfigMonthly()),
    # 'titan_patch': lambda cfg: build_titan_patch(cfg or TitanConfigPatchMonthly()),
    # "patchtst_base": lambda cfg: build_patchTST_base(cfg or PatchTSTConfigMonthly()),
    # "patchtst_quantile": lambda cfg: build_patchTST_quantile(cfg or PatchTSTConfigMonthly()),
}
loaded = load_model_dict(save_dir, builders, device = device)



In [ ]:
%load_ext autoreload
%autoreload 2

import importlib, modeling_module.utils.plot_utils as pu
import modeling_module.training.forecaster as fo
importlib.reload(pu)
importlib.reload(fo)

def my_exo_cb(start_idx: int, Hm: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    # exo_dim = 2 (sin, cos)
    return fo.make_calendar_exo(start_idx, Hm, period=52, device=device)

pu.plot_27w(
    models=loaded,           # {"PatchMixer": pm_model, "Titan": ti_model, ...}
    loader=val_loader,       # (xb, yb[, part_ids])
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="val",              # ← 검증 모드
    max_plots=5,
    out_dir=None,
    show=True,
    future_exo_cb=my_exo_cb
)

In [ ]:
import importlib
from modeling_module.utils import temporal_expander as te
from modeling_module.models.PatchMixer import PatchMixer as pm
model = loaded["patchmixer_base"].to(device).eval()
x1 = next(iter(val_loader))[0][:1].to(device)   # [1, L, C] or [1, L]

with torch.no_grad():
    x_n   = model.revin(x1, 'norm')
    z     = model.backbone(x_n)
    x_bhf = model.expander(z)
    x_bhf_n = model.pre_ln(x_bhf)
    resid = model.head(x_bhf_n).squeeze(-1)

    # base/alpha
    H = model.horizon
    t = torch.linspace(-1, 1, H, device=z.device).unsqueeze(0)
    b = model.base_head_b(z); m = model.base_head_m(z)
    base  = b + m * t
    alpha = torch.sigmoid(model.base_gate(z)).expand(-1, H)

    # gate
    xg = model.gate_ln(x_bhf_n).transpose(1,2)
    g1 = model.gate_act(model.gate_conv_3(xg))
    g2 = model.gate_act(model.gate_conv_5(xg))
    g3 = model.gate_act(model.gate_conv_d3(xg))
    glogit = model.gate_reduce(torch.cat([g1,g2,g3], 1)).transpose(1,2).squeeze(-1)
    tau = torch.linspace(-1,1,H,device=z.device).view(1,H).expand(z.size(0),H)
    glogit = model.g_gain * ((glogit + model.tau_weight * tau + model.g_bias) / model.gate_temp)
    gate = torch.sigmoid(torch.clamp(glogit, -model.g_logit_clip, model.g_logit_clip))

print("resid var(H) mean:", resid.var(dim=1).mean().item())
print("gate mean/std:", gate.mean().item(), gate.std().item())
print("alpha mean:", alpha.mean().item())

te  = importlib.reload(te)   # Expander 변경 반영
pm  = importlib.reload(pm)   # PatchMixer 변경 반영
old = loaded['patchmixer_base']        # 예전 인스턴스
sd  = old.state_dict()                  # 가중치만 백업
cfg = pm_base_config  # 같은 horizon/enc_in 등
exo_dim = getattr(old, 'exo_dim', 0)

new_model = pm.BaseModel(configs=cfg, exo_dim=exo_dim)

new_model = new_model.to(device).eval()
loaded['patchmixer_base'] = new_model
model = new_model   # 이후엔 이 객체로 사용

model.eval(); model.final_nonneg = False

# def slope_mean(v):  # v: [B,H]
#     return float((v[:, -1] - v[:, 0]).mean())
#
# with torch.no_grad():
#     x_n  = model.revin(x1, 'norm')
#     z    = model.backbone(x_n)
#     H    = model.horizon
#     t    = torch.linspace(-1, 1, H, device=z.device).unsqueeze(0)
#
#     b = model.base_head_b(z); m = model.base_head_m(z)
#     base  = b + m * t
#     alpha = torch.sigmoid(model.base_gate(z)).expand(-1, H)
#
#     xbhf   = model.expander(z)
#     xbhf_n = model.pre_ln(xbhf)
#     resid  = model.head(xbhf_n).squeeze(-1)
#     resid  = resid - resid.mean(dim=1, keepdim=True)   # 추가한 줄과 맞춤
#     resid  = resid * float(model.resid_scale)
#
#     xg = model.gate_ln(xbhf_n).transpose(1,2)
#     g1 = model.gate_act(model.gate_conv_3(xg))
#     g2 = model.gate_act(model.gate_conv_5(xg))
#     g3 = model.gate_act(model.gate_conv_d3(xg))
#     gcat   = model.gate_do(torch.cat([g1,g2,g3], dim=1))
#     glogit = model.gate_reduce(gcat).transpose(1,2).squeeze(-1)
#     # τ 영향 제거한 상태라면 tau_weight=0
#     tau    = torch.linspace(-1, 1, H, device=z.device).view(1,H).expand(z.size(0),H)
#     glogit = model.g_gain * ((glogit + model.tau_weight * tau + model.g_bias) / model.gate_temp)
#     glogit = torch.clamp(glogit, -model.g_logit_clip, model.g_logit_clip)
#     gate   = torch.sigmoid(glogit)
#     gate   = gate - gate.mean(dim=1, keepdim=True) + 0.5
#     gate   = torch.clamp(gate, 0.05, 0.95)
#
#     resid_part = (1.0 - alpha) * (gate * resid)
#     y_n = alpha * base + resid_part
#
#     print("alpha mean:", float(alpha.mean()))
#     print("m mean:", float(m.mean()))
#     print("base slope(mean):", slope_mean(base))
#     print("resid_part slope(mean):", slope_mean(resid_part))
#     print("y slope(mean):", slope_mean(y_n))



In [ ]:
@torch.no_grad()
def check_exo_compat(model, cfg, future_exo_cb, x_sample: torch.Tensor):
    # 1) 기본 정보
    mdl_exo = int(getattr(model, "exo_dim", 0))
    cfg_exo = int(getattr(cfg, "exo_dim", 0))
    H = int(getattr(model, "horizon", 0)) or int(getattr(model, "output_horizon", 0))
    print(f"[EXO] model.exo_dim={mdl_exo}, cfg.exo_dim={cfg_exo}, horizon={H}")

    # 2) 콜백이 만든 exo의 shape
    exo = None
    if future_exo_cb is not None and H > 0:
        exo = future_exo_cb(0, H, device=x_sample.device)  # (H, exo_dim) 예상
        if exo.dim() == 2:  # (H, D) -> (B,H,D)로 확장 예행연습
            exo = exo.unsqueeze(0).expand(x_sample.size(0), -1, -1)
        print(f"[EXO] cb exo shape: {tuple(exo.shape)} (expect [B,H,D])")
    else:
        print("[EXO] future_exo_cb=None or horizon=0 → 콜백 점검 불가")

    # 3) 정합성 체크
    if mdl_exo > 0:
        assert exo is not None, "model.exo_dim>0 인데 future_exo가 None 입니다."
        assert exo.size(1) == H, f"exo H({exo.size(1)}) != model.horizon({H})"
        assert exo.size(2) == mdl_exo, f"exo_dim({exo.size(2)}) != model.exo_dim({mdl_exo})"
        if cfg_exo != mdl_exo:
            print(f"[WARN] cfg.exo_dim({cfg_exo}) != model.exo_dim({mdl_exo}) → 설정/모델 불일치")
    else:
        if exo is not None:
            print("[INFO] model.exo_dim==0 이지만 exo가 생성되었습니다(무시되거나 오류 유발 가능).")

    # 4) 드라이런
    try:
        _ = model(x_sample, future_exo=exo)
        print("[EXO] forward(future_exo=exo) OK")
    except TypeError:
        # 모델 시그니처가 future_exo를 안 받을 수도 있으므로 일반 호출도 검사
        _ = model(x_sample)
        print("[EXO] forward(x) OK (모델이 future_exo 인자를 받지 않음)")


In [ ]:
check_exo_compat(model, cfg, future_exo_cb=calendar_sin_cos, x_sample=x1)


In [ ]:
# plan_yyyyww 기준으로 과거 히스토리 + 27주 예측을 그림
# pu.plot_27w(
#     models=loaded,
#     loader=inference_loader,  # (xb, part_ids)
#     device="cuda",
#     mode="infer",             # ← 추론 모드
#     plan_yyyyww=202544,       # 앵커 주차(예: 2025년 44주)
#     max_plots=10,
#     out_dir=None,
#     show=True,
#     future_exo_cb=my_exo_cb,
#     # 필요하면 GT를 불러오는 콜백(옵션)
#     truth_cb=None            # 또는 truth_cb(part_id, plan_dt, H=27, 'week') -> np.ndarray
# )

In [ ]:
# pu.plot_120m(
#     models=loaded,
#     loader=val_loader,       # (xb, yb[, part_ids])
#     device="cuda",
#     mode="val",
#     max_plots=5,
#     out_dir=None,
#     show=True,
#     future_exo_cb=my_exo_cb
# )

In [ ]:
# pu.plot_120m(
#     models=loaded,
#     loader=inference_loader,  # (xb, part_ids)
#     device="cuda",
#     mode="infer",
#     plan_yyyymm=202510,       # 앵커 월(예: 2025년 10월)
#     max_plots=10,
#     out_dir=None,
#     show=True,
#     future_exo_cb=my_exo_cb,
#     # 필요하면 월 단위 GT 조회 콜백
#     truth_cb=None            # 또는 truth_cb(part_id, plan_dt, H=120, 'month') -> np.ndarray
# )